# 4. SBI Parameter Recovery & Inference

Fit the BE model to behavioural data using Simulation-Based Inference (SBI).

**Workflow:**
1. Single-session recovery: validate that SBI can recover known params
2. Multi-session recovery: validate temporal structure (GP-linked eta)
3. Real data inference (once validated on synthetic)

**Prerequisites:**
- Notebook 2: stat selection validated
- `pip install sbi torch`

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
import warnings, time
warnings.filterwarnings('ignore')

from Models.BE_core import BEParams, BEState, BEModel
from Analysis.summary_stats import (
    compute_summary_stats, get_stat_names_expanded,
)
from Data.structures import (
    generate_synthetic_animal, sample_stimuli, stimulus_to_category,
    param_trajectory_naive_to_expert,
)

try:
    import torch
    import sbi
    from sbi.inference import SNPE
    from sbi.utils import BoxUniform
    SBI_AVAILABLE = True
    print(f"SBI version: {sbi.__version__}, torch: {torch.__version__}")
except ImportError:
    SBI_AVAILABLE = False
    print("WARNING: sbi/torch not installed. pip install sbi torch")

## 1. Single-Session Parameter Recovery

The simplest test: can SBI recover parameters from a single session?
This validates that the summary stats carry enough information.

In [ ]:
# --- Configuration ---
# Use the stats validated in notebook 2
STAT_NAMES = [
    'accuracy', 'psychometric', 'recency', 'win_stay',
    'stimulus_sensitivity', 'logistic_history',
]
expanded = get_stat_names_expanded(STAT_NAMES)
n_stats = len(expanded)
print(f"Using {n_stats} stats: {expanded}")

N_TRIALS = 300
BURN_IN = 100
N_SBI_SIMS = 10_000  # Training simulations (increase for better results)
SEED = 42

In [ ]:
# --- Build simulator for SBI ---
def simulate_for_sbi(theta):
    """
    SBI simulator: takes parameter tensor, returns summary stat tensor.
    
    theta: [sigma_percep, A_repulsion, eta_learning, eta_relax]
    """
    # Extract params
    if isinstance(theta, torch.Tensor):
        theta = theta.numpy()
    
    sigma_percep, A_repulsion, eta_learning, eta_relax = theta
    
    params = BEParams(
        sigma_percep=float(sigma_percep),
        A_repulsion=float(A_repulsion),
        eta_learning=float(eta_learning),
        eta_relax=float(eta_relax),
    )
    
    rng = np.random.default_rng()
    
    # Burn-in
    state = BEState.initial_uniform()
    burn_stim = rng.uniform(-1.0, 1.0, BURN_IN)
    burn_cats = stimulus_to_category(burn_stim)
    _, _, state, _ = BEModel.simulate_session(params, state, burn_stim, burn_cats, rng)
    
    # Simulate
    stimuli = sample_stimuli(N_TRIALS, 'Uniform', rng)
    categories = stimulus_to_category(stimuli)
    choices, _, _, _ = BEModel.simulate_session(params, state, stimuli, categories, rng)
    
    # Compute stats
    try:
        stats = compute_summary_stats(
            choices, stimuli, categories,
            stat_names=STAT_NAMES, return_dict=False,
        )
        return torch.tensor(stats, dtype=torch.float32)
    except Exception:
        return torch.full((n_stats,), float('nan'))

if SBI_AVAILABLE:
    print("Simulator ready.")

In [ ]:
# --- Define prior ---
if SBI_AVAILABLE:
    bounds = BEParams.get_bounds()
    prior = BoxUniform(
        low=torch.tensor([bounds['sigma_percep'][0], bounds['A_repulsion'][0],
                          bounds['eta_learning'][0], bounds['eta_relax'][0]]),
        high=torch.tensor([bounds['sigma_percep'][1], bounds['A_repulsion'][1],
                           bounds['eta_learning'][1], bounds['eta_relax'][1]]),
    )
    print(f"Prior bounds: {bounds}")

In [ ]:
# --- Run SBI training ---
if SBI_AVAILABLE:
    print(f"Simulating {N_SBI_SIMS} training examples...")
    t0 = time.time()
    
    inference = SNPE(prior=prior)
    
    # Simulate training data
    theta_train = prior.sample((N_SBI_SIMS,))
    x_train = torch.stack([simulate_for_sbi(t) for t in theta_train])
    
    # Remove NaN simulations
    valid = ~torch.any(torch.isnan(x_train), dim=1)
    theta_train = theta_train[valid]
    x_train = x_train[valid]
    print(f"  {valid.sum()}/{N_SBI_SIMS} valid ({time.time()-t0:.0f}s)")
    
    # Train
    print("Training neural posterior...")
    density_estimator = inference.append_simulations(theta_train, x_train).train()
    posterior = inference.build_posterior(density_estimator)
    print(f"  Done ({time.time()-t0:.0f}s total)")

### Recovery test on synthetic data

In [ ]:
# --- Generate test data with known parameters ---
if SBI_AVAILABLE:
    N_TEST = 50
    rng_test = np.random.default_rng(123)
    
    true_params_list = []
    posterior_means = []
    posterior_stds = []
    
    print(f"Testing recovery on {N_TEST} synthetic sessions...")
    for i in range(N_TEST):
        # Sample true params
        true_theta = prior.sample((1,)).squeeze()
        true_params_list.append(true_theta.numpy())
        
        # Generate observed data
        x_obs = simulate_for_sbi(true_theta)
        
        if torch.any(torch.isnan(x_obs)):
            posterior_means.append(np.full(4, np.nan))
            posterior_stds.append(np.full(4, np.nan))
            continue
        
        # Get posterior
        samples = posterior.sample((1000,), x=x_obs)
        posterior_means.append(samples.mean(dim=0).numpy())
        posterior_stds.append(samples.std(dim=0).numpy())
    
    true_params_arr = np.array(true_params_list)
    post_means_arr = np.array(posterior_means)
    post_stds_arr = np.array(posterior_stds)
    print("Done.")

In [ ]:
# --- Recovery plot ---
if SBI_AVAILABLE:
    param_labels = ['sigma_percep', 'A_repulsion', 'eta_learning', 'eta_relax']
    
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    
    for p, (ax, label) in enumerate(zip(axes.flat, param_labels)):
        true_vals = true_params_arr[:, p]
        post_vals = post_means_arr[:, p]
        post_err = post_stds_arr[:, p]
        
        valid = ~np.isnan(post_vals)
        
        ax.errorbar(true_vals[valid], post_vals[valid], yerr=post_err[valid],
                    fmt='o', markersize=4, alpha=0.5, color='#1976d2',
                    elinewidth=0.5, capsize=0)
        
        # Identity line
        lims = [bounds[label][0], bounds[label][1]]
        ax.plot(lims, lims, 'k--', linewidth=1)
        
        # Correlation
        rho, pval = spearmanr(true_vals[valid], post_vals[valid])
        ax.set_title(f'{label}\nrho={rho:.3f}, p={pval:.2e}')
        ax.set_xlabel('True')
        ax.set_ylabel('Posterior mean')
        ax.set_xlim(lims)
        ax.set_ylim(lims)
        ax.set_aspect('equal')
    
    plt.suptitle('Single-session parameter recovery', fontsize=12)
    plt.tight_layout()
    plt.show()
    
    # Summary
    for p, label in enumerate(param_labels):
        valid = ~np.isnan(post_means_arr[:, p])
        rho, _ = spearmanr(true_params_arr[valid, p], post_means_arr[valid, p])
        rmse = np.sqrt(np.mean((true_params_arr[valid, p] - post_means_arr[valid, p])**2))
        print(f"  {label:<16s}: rho={rho:.3f}, RMSE={rmse:.4f}")

## 2. Multi-Session Recovery (GP-linked eta)

The real use case: fit a trajectory of sessions with temporally-linked
parameters. eta_learning varies across sessions (GP prior), while
sigma_percep and A_repulsion are constant.

This requires the SBIFitter infrastructure from the codebase.

In [ ]:
# --- Multi-session setup ---
# This section uses the SBIFitter class from the codebase.
# If not yet refactored, use the direct approach below.

if SBI_AVAILABLE:
    print("Multi-session SBI requires the SBIFitter class.")
    print("Once the inference interface is consolidated (step 4 in the roadmap),")
    print("add the multi-session fitting code here.")
    print()
    print("The key idea:")
    print("  - sigma_percep, A_repulsion: ConstantLink (1 dim each, uniform prior)")
    print("  - eta_learning: GPLink (N_sessions dims, GP prior with learned lengthscale)")
    print("  - eta_relax: ConstantLink (1 dim, uniform prior)")
    print()
    print("Total theta dimensions: 2 + N_sessions + 1 = N_sessions + 3")
    print("Summary stats: N_sessions * n_stats_per_session")
    print()
    print("The GP prior on eta enforces smoothness, which helps break")
    print("the parameter degeneracy identified in notebook 2.")

## 3. Posterior Predictive Checks

Given fitted parameters, simulate new data and compare to observed.
This validates that the model can reproduce the observed behaviour.

In [ ]:
# --- Posterior predictive check for a single session ---
if SBI_AVAILABLE:
    # Pick a test case
    test_idx = 0
    true_theta = true_params_list[test_idx]
    x_obs = simulate_for_sbi(torch.tensor(true_theta, dtype=torch.float32))
    
    # Draw posterior samples and simulate
    post_samples = posterior.sample((200,), x=x_obs)
    
    x_pred = []
    for s in post_samples[:200]:
        x = simulate_for_sbi(s)
        if not torch.any(torch.isnan(x)):
            x_pred.append(x.numpy())
    x_pred = np.array(x_pred)
    
    # Compare observed vs predicted stats
    fig, axes = plt.subplots(2, int(np.ceil(n_stats/2)), figsize=(16, 8))
    axes = axes.flat
    
    for s, ax in enumerate(axes):
        if s >= n_stats:
            ax.set_visible(False)
            continue
        
        ax.hist(x_pred[:, s], bins=20, density=True, alpha=0.7,
                color='#1976d2', label='Posterior predictive')
        ax.axvline(x_obs[s].item(), color='red', linewidth=2,
                   linestyle='--', label='Observed')
        ax.set_title(expanded[s], fontsize=8)
        ax.tick_params(labelsize=7)
        if s == 0:
            ax.legend(fontsize=7)
    
    plt.suptitle('Posterior predictive check', fontsize=12)
    plt.tight_layout()
    plt.show()

---

**Summary of the notebook pipeline:**
1. **Notebook 1**: Generate and visualise synthetic data
2. **Notebook 2**: Select informative summary statistics  
3. **Notebook 3**: Discover behavioural states with HMM
4. **Notebook 4** (this): Fit model parameters with SBI

For real data: replace `generate_synthetic_animal` with `load_animal` /
`load_experiment` from Data.structures, and the rest of the pipeline
should work identically.